## Lecture 3

### MNIST Dataset

This notebook builds a custom PyTorch loss function -- the Cauchy-Schwarz
Divergence -- and trains the `LeNet5` model from Lecture 1's notebook on
MNIST with it. Since this is a separate notebook file, `LeNet5` is
redefined below (identical to Lecture 1) so this one runs standalone.

In [1]:
import torch
import torchvision

In [2]:
data_train = torchvision.datasets.MNIST(
    './data/mnist',
    train=True,
    download=True,
    transform=torchvision.transforms.Compose([
        torchvision.transforms.Pad(2),
        torchvision.transforms.ToTensor()
    ])
)

data_test = torchvision.datasets.MNIST(
    './data/mnist',
    train=False,
    download=True,
    transform=torchvision.transforms.Compose([
        torchvision.transforms.Pad(2),
        torchvision.transforms.ToTensor()
    ])
)

Next we specify the batch size to be `32`.

In [3]:
BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(
    data_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

test_loader = torch.utils.data.DataLoader(
    data_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

### GPU Support

The course hardcodes `if torch.cuda.is_available(): device = torch.device('cuda')`.
On this Mac we prefer Apple's Metal backend (`mps`) and fall back through
`cuda` to `cpu`, matching the pattern used throughout the other course
notebooks in this repo.

In [4]:
device = torch.device('cpu')
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')

device

device(type='mps')

### `LeNet5` (from Lecture 1)

Same architecture as Lecture 1's notebook -- redefined here since this is a
separate file.

In [5]:
class LeNet5(torch.nn.Module):

    def __init__(self):

        super(LeNet5, self).__init__()

        self.convnet = torch.nn.Sequential(
            # Conv Block 1
            torch.nn.Conv2d(
                in_channels=1,
                out_channels=6,
                kernel_size=(5, 5),
                stride=1,
                bias=True
            ),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(
                kernel_size=(2, 2),
                stride=2
            ),
            # Conv Block 2
            torch.nn.Conv2d(
                in_channels=6,
                out_channels=16,
                kernel_size=(5, 5),
                stride=1,
                bias=True
            ),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(
                kernel_size=(2, 2),
                stride=2
            ),
            # Conv Block 3
            torch.nn.Conv2d(
                in_channels=16,
                out_channels=120,
                kernel_size=(5, 5),
                stride=1,
                bias=True
            ),
            torch.nn.ReLU(),
        )

        self.fcn = torch.nn.Sequential(
            # Fully Connected Layer 1
            torch.nn.Linear(
                in_features=120,
                out_features=84,
                bias=True
            ),
            torch.nn.ReLU(),
            # Classifier Layer 2
            torch.nn.Linear(
                in_features=84,
                out_features=10,
                bias=True
            ),
            torch.nn.Softmax(dim=1)
        )

    def forward(self, batch):
        ret = self.convnet(batch)
        ret = ret.view(batch.size(0), -1)
        ret = self.fcn(ret)
        return ret

### Objective Functions in Deep Learning

For background on why custom, similarity-based objective functions like
this one are useful in deep learning (e.g. cosine similarity):
[youtu.be/N9mcfsHac1U](https://youtu.be/N9mcfsHac1U)

### A Short Introduction to Entropy, Cross-Entropy and KL-Divergence

Another influential resource, oriented toward the more standard
entropy-based loss functions, by Aurelien Geron:
[youtu.be/ErfnhcEV1O8](https://youtu.be/ErfnhcEV1O8)

### Cauchy-Schwarz Divergence

This custom loss is built from two pieces:

- `encodeOneHot(torch_tensor)` converts a batch of integer class labels
  into one-hot vectors -- note it moves the tensor to CPU
  (`.cpu().numpy()`) to build the one-hot matrix with NumPy, then moves the
  result back onto `device` before returning.
- `CSD` (`torch.nn.Module`) implements the Cauchy-Schwarz divergence itself
  as a similarity-based loss between the network's output and the
  one-hot-encoded target.

In [6]:
import numpy as np


def encodeOneHot(torch_tensor):
    a = torch_tensor.cpu().numpy()

    b = np.zeros((a.size, 10))

    b[np.arange(a.size), a] = 1

    return torch.from_numpy(b).float().to(device)


class CSD(torch.nn.Module):

    def __init__(self):

        super(CSD, self).__init__()

    def forward(self, outputs, target):

        y = encodeOneHot(target)

        nom = torch.sum(torch.mm(outputs, y.t()), dim=1)

        denom = torch.norm(outputs, 2) * torch.norm(y, 2)

        return torch.mean(-1 * torch.log(nom / denom))

### Model Training

`network = LeNet5().to(device)` moves the model onto the Metal (`mps`)
device selected above -- everything else follows the same pattern already
established (`images, labels = images.to(device), labels.to(device)` inside
the loop).

In [7]:
network = LeNet5().to(device)
optimizer = torch.optim.Adam(network.parameters(), lr=0.0005)
criterion = CSD()

In order to perform the model training we define an epoch loop with methods
efficiently implemented by PyTorch -- now we implement the inner loop
through each batch. We convert the batch to tensors and move the data to
the GPU, and we do our usual gradient descent.

In [8]:
epochs = 128
steps = len(train_loader) // BATCH_SIZE

network.train(True)

for e in range(epochs):

    epoch_loss = 0

    performed_steps = 0

    for i, (images, labels) in enumerate(train_loader):

        if i == steps:
            break

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = network(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        epoch_loss += loss

        performed_steps += 1

    assert performed_steps == steps, "steps: {} != {}".format(steps, performed_steps)

    print("Epoch: {}, Loss: {:0.4f}".format(
        e + 1,
        epoch_loss / steps
    ))

Epoch: 1, Loss: 1.1555
Epoch: 2, Loss: 1.1504
Epoch: 3, Loss: 1.1490
Epoch: 4, Loss: 1.1524
Epoch: 5, Loss: 1.1506
Epoch: 6, Loss: 1.1513
Epoch: 7, Loss: 1.1511
Epoch: 8, Loss: 1.1506
Epoch: 9, Loss: 1.1506
Epoch: 10, Loss: 1.1521
Epoch: 11, Loss: 1.1522
Epoch: 12, Loss: 1.1506
Epoch: 13, Loss: 1.1495
Epoch: 14, Loss: 1.1510
Epoch: 15, Loss: 1.1490
Epoch: 16, Loss: 1.1507
Epoch: 17, Loss: 1.1496
Epoch: 18, Loss: 1.1517
Epoch: 19, Loss: 1.1492
Epoch: 20, Loss: 1.1508
Epoch: 21, Loss: 1.1488
Epoch: 22, Loss: 1.1501
Epoch: 23, Loss: 1.1490
Epoch: 24, Loss: 1.1521
Epoch: 25, Loss: 1.1527
Epoch: 26, Loss: 1.1505
Epoch: 27, Loss: 1.1482
Epoch: 28, Loss: 1.1486
Epoch: 29, Loss: 1.1492
Epoch: 30, Loss: 1.1499
Epoch: 31, Loss: 1.1502
Epoch: 32, Loss: 1.1506
Epoch: 33, Loss: 1.1504
Epoch: 34, Loss: 1.1507
Epoch: 35, Loss: 1.1505
Epoch: 36, Loss: 1.1495
Epoch: 37, Loss: 1.1478
Epoch: 38, Loss: 1.1512
Epoch: 39, Loss: 1.1494
Epoch: 40, Loss: 1.1495
Epoch: 41, Loss: 1.1477
Epoch: 42, Loss: 1.1469
E

You can see here that I have trained this network for `128` epochs and the
loss has been decreasing from the weights.

### Model Evaluation

And next we calculate both the loss and the accuracy.

In [9]:
avg_loss = 0
avg_acc = 0

network.train(False)

with torch.no_grad():

    steps = 0

    for images, labels in test_loader:

        images, labels = images.to(device), labels.to(device)

        outputs = network(images)

        avg_loss += criterion(outputs, labels)

        _, preds = torch.max(outputs, 1)
        avg_acc += preds.eq(labels).sum().item()

        steps += 1

    print("Loss: {:0.2f}, Acc: {:.2%}".format(
        avg_loss / steps,
        avg_acc / (steps * BATCH_SIZE)
    ))

Loss: 1.15, Acc: 90.78%


### Read More

- Robert Jenssen, Jose C. Principe, Deniz Erdogmus, Torbjorn Eltoft, The
  Cauchy-Schwarz divergence and Parzen windowing: Connections to graph
  theory and Mercer kernels, Journal of the Franklin Institute, Volume 343,
  Issue 6, 2006, Pages 614-629, ISSN 0016-0032,
  [doi.org/10.1016/j.jfranklin.2006.03.018](https://doi.org/10.1016/j.jfranklin.2006.03.018).
- Jenssen, R., Eltoft, T., Erdogmus, D. et al. J VLSI Sign Process Syst
  Sign Image Video Technol (2006) 45: 49.
  [doi.org/10.1007/s11265-006-9771-8](https://doi.org/10.1007/s11265-006-9771-8)
- Janocha, K., & Czarnecki, W. (2017). On Loss Functions for Deep Neural
  Networks in Classification. CoRR, abs/1702.05659.
  [arxiv.org/abs/1702.05659](https://arxiv.org/abs/1702.05659)
- Lecun, Yann & Bottou, Leon & Bengio, Y & Haffner, Patrick. (1998).
  Gradient-Based Learning Applied to Document Recognition. Proceedings of
  the IEEE. 86. 2278-2324. 10.1109/5.726791.
  [yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf)

### Course Discussions

**Slack Channel:** the invite link shown in the screenshot is truncated by
the video frame width and cannot be fully reconstructed --
`https://mqubits.slack.com/join/shared_invite/enQtNjU2MTQ3ODgxMjY3LTgyMGM3MzFmOTQ3MTQ3OT...`
(cut off). If you need the working invite, it's worth checking the course's
official resources/discussion page directly rather than relying on this
partial capture.

**Google Colab (original notebook):**
[colab.research.google.com/drive/1BON2RGHeRqS1tM3raz6HSpO8Xfz5-vYy](https://colab.research.google.com/drive/1BON2RGHeRqS1tM3raz6HSpO8Xfz5-vYy)

### Assignment

**Reading Assignment**

- Janocha, K., & Czarnecki, W. (2017). On Loss Functions for Deep Neural
  Networks in Classification. CoRR, abs/1702.05659.
  [arxiv.org/abs/1702.05659](https://arxiv.org/abs/1702.05659)

**Coding Exercise**

Implement Custom Loss Function -- pick one of the other loss functions
discussed (e.g. cosine similarity, cross-entropy/KL-divergence -- see the
two linked videos above) and implement it as a custom `torch.nn.Module`,
the same way `CSD` was implemented here, based on what you've learned in
this section.